# Laboratorio 10: Vectorización, Indexación y Reconocimiento de Rostros
## Base de Datos II - Profesor Heider Sanchez
### P1: Generar los vectores característicos

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import face_recognition
import psycopg2

DATASET_PATH = "/home/tkaos/UTEC/C5/bd2/sol/data/lfw_funneled"

In [ ]:
# Cargar dataset (P0)
coleccion = []
for path in glob.iglob(os.path.join(DATASET_PATH, "**", "*.jpg")):
    person = path.split("/")[-2]
    coleccion.append({"person": person, "path": path})

coleccion = pd.DataFrame(coleccion)
print(f"Total de imagenes: {len(coleccion)}")
coleccion.head(10)

In [ ]:
def generate_face_embeddings(coleccion, N):
    """
    Genera embeddings faciales para las primeras N imagenes de la coleccion.
    
    Args:
        coleccion: DataFrame con columnas 'person' y 'path'
        N: Numero de rostros a procesar
    
    Returns:
        Lista de diccionarios con person, path y embedding
    """
    results = []
    for i in range(min(N, len(coleccion))):
        row = coleccion.iloc[i]
        filename = row.path
        
        image = face_recognition.load_image_file(filename)
        face_encodings = face_recognition.face_encodings(image)
        
        if face_encodings:
            embedding = face_encodings[0]
            results.append({
                "person": row.person,
                "path": filename,
                "embedding": embedding.tolist()
            })
        else:
            print(f"No se detecto rostro en: {filename}")
    
    return results

In [ ]:
# Configurar conexion a PostgreSQL con pgvector
DB_CONFIG = {
    "dbname": "postgres",
    "user": "postgres",
    "password": "123456",
    "host": "localhost",
    "port": 5433
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

# Habilitar extension pgvector
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Crear tabla si no existe
cur.execute("""
    CREATE TABLE IF NOT EXISTS face_embeddings (
        id SERIAL PRIMARY KEY,
        name TEXT,
        path TEXT,
        embedding VECTOR(128)
    );
""")
conn.commit()
print("Tabla face_embeddings lista.")

In [ ]:
# Generar embeddings y guardar en PostgreSQL
N = 100  # Cambiar segun necesidad
embeddings_data = generate_face_embeddings(coleccion, N)
print(f"Embeddings generados: {len(embeddings_data)}")

# Insertar en BD
for item in embeddings_data:
    cur.execute(
        "INSERT INTO face_embeddings (name, path, embedding) VALUES (%s, %s, %s)",
        (item["person"], item["path"], item["embedding"])
    )
conn.commit()
print(f"Insertados {len(embeddings_data)} registros en face_embeddings.")

In [ ]:
# Verificar datos almacenados
cur.execute("SELECT COUNT(*) FROM face_embeddings;")
count = cur.fetchone()[0]
print(f"Total de registros en BD: {count}")

cur.execute("SELECT id, name, path FROM face_embeddings LIMIT 5;")
for row in cur.fetchall():
    print(row)

# Cerrar conexion
cur.close()
conn.close()